# YOLOv8 모델 최적화: PyTorch → ONNX → TensorRT

## 목표
- YOLOv8 CARLA 파인튜닝 모델을 ONNX / TensorRT FP16으로 변환
- 포맷별 추론 속도(FPS) / 지연시간(latency) / 정확도(mAP) 정량 비교
- 실시간 자율주행 시스템에서 어느 포맷이 적합한지 근거 제시

## 핵심 개념
- **ONNX**: 프레임워크 독립 IR — PyTorch 연산 그래프를 표준 포맷으로 직렬화
- **TensorRT**: NVIDIA GPU 특화 추론 엔진 — 레이어 퓨전 + FP16/INT8 양자화로 속도 극대화
- **FP16 (half precision)**: 가중치를 16-bit로 줄여 메모리 절반 + 텐서코어 활용 → 2~4× 속도 향상
- **레이어 퓨전(layer fusion)**: Conv+BN+ReLU를 하나의 커널로 합쳐 메모리 왕복 감소

## 비교 대상
| 포맷 | 디바이스 | 정밀도 | 예상 속도 |
|------|----------|--------|-----------|
| PyTorch | GPU | FP32 | 기준 |
| PyTorch | GPU | FP16 | ~1.5× |
| ONNX | CPU | FP32 | 느림 |
| ONNX | GPU (CUDAExecutionProvider) | FP32 | ~1.2× |
| TensorRT | GPU | FP16 | ~3~5× |

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

## 셀 1: 환경 확인 및 패키지 설치

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'], check=True)

def pip_uninstall(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', *pkgs, '-y', '-q'],
                   capture_output=True)

# onnxruntime-gpu: onnxruntime(CPU)와 공존 불가 → 교체 필요
try:
    import onnxruntime as ort
    if 'CUDAExecutionProvider' not in ort.get_available_providers():
        print('onnxruntime(CPU) 감지 → onnxruntime-gpu로 교체 중...')
        pip_uninstall('onnxruntime')
        pip_install('onnxruntime-gpu')
        print('교체 완료 — 아래 import 다시 실행')
    else:
        print(f'onnxruntime-gpu 이미 설치됨: {ort.__version__}')
except ImportError:
    print('onnxruntime-gpu 설치 중...')
    pip_install('onnxruntime-gpu')

# TensorRT Python 바인딩
try:
    import tensorrt as trt
    print(f'TensorRT: {trt.__version__}')
except ImportError:
    print('TensorRT 설치 중... (CUDA 12.x 전용, 수 분 소요)')
    pip_install('tensorrt')

# 재import (설치 직후엔 importlib.reload 필요)
import importlib, onnxruntime as ort
importlib.reload(ort)

import torch
print(f'PyTorch:       {torch.__version__}')
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')
print(f'GPU:           {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음"}')
print(f'ONNX Runtime:  {ort.__version__}')
print(f'ORT Providers: {ort.get_available_providers()}')

try:
    import tensorrt as trt
    print(f'TensorRT:      {trt.__version__}')
    TRT_AVAILABLE = True
except ImportError:
    print('TensorRT:      미설치 (ONNX까지만 벤치마크)')
    TRT_AVAILABLE = False

## 셀 2: 모델 및 테스트 데이터 준비

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import numpy as np
import cv2

# ─── 모델 경로 ─────────────────────────────────────────
# CARLA 파인튜닝 best.pt 우선, 없으면 공개 yolov8n.pt 사용
CARLA_PT  = Path('../../phase4_carla/finetuning/runs/carla_finetune/weights/best.pt')
COCO_PT   = Path('../../phase1_basics/detection/yolov8n.pt')
MODEL_PT  = CARLA_PT if CARLA_PT.exists() else COCO_PT

print(f'사용 모델: {MODEL_PT}')

# ─── 테스트 이미지 로드 ────────────────────────────────
IMG_DIR = Path('../../phase4_carla/data_collection/carla_dataset/images')
img_paths = sorted(IMG_DIR.glob('*.jpg'))[:50]  # 벤치마크용 50장

if not img_paths:
    # 합성 이미지로 대체
    print('실제 이미지 없음 → 합성 이미지 사용')
    dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
    tmp = Path('/tmp/dummy.jpg')
    cv2.imwrite(str(tmp), dummy)
    img_paths = [tmp] * 50

print(f'벤치마크 이미지: {len(img_paths)}장')

# ─── 입력 텐서 준비 (640×640) ──────────────────────────
sample_img = cv2.imread(str(img_paths[0]))
sample_img = cv2.resize(sample_img, (640, 640))
sample_tensor = torch.from_numpy(sample_img).permute(2,0,1).float().unsqueeze(0) / 255.0
sample_np = sample_tensor.numpy()

print(f'입력 텐서 shape: {sample_tensor.shape}  dtype: {sample_tensor.dtype}')

## 셀 3: PyTorch 기준선 벤치마크 (FP32 / FP16)

In [ ]:
import time

WARMUP = 10   # 워밍업 횟수 (GPU 초기화 제거)
REPEAT = 50   # 측정 횟수

def benchmark_torch(model_pt, half=False, device='cuda', n_warmup=WARMUP, n_repeat=REPEAT):
    """PyTorch 추론 벤치마크 → FPS, latency 반환"""
    model = YOLO(str(model_pt))
    if half:
        model.model.half()
    model.model.to(device).eval()

    img = sample_tensor.to(device)
    if half:
        img = img.half()

    # 워밍업
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model.model(img)
    torch.cuda.synchronize()

    # 측정
    latencies = []
    with torch.no_grad():
        for _ in range(n_repeat):
            t0 = time.perf_counter()
            _ = model.model(img)
            torch.cuda.synchronize()
            latencies.append((time.perf_counter() - t0) * 1000)  # ms

    lat_mean = np.mean(latencies)
    lat_p99  = np.percentile(latencies, 99)
    fps      = 1000 / lat_mean
    label    = f'PyTorch FP{"16" if half else "32"} ({device.upper()})'
    print(f'{label:<35} | {lat_mean:6.2f} ms | {fps:7.1f} FPS | P99: {lat_p99:.1f} ms')
    return {'label': label, 'latency_ms': lat_mean, 'fps': fps, 'p99_ms': lat_p99}

print(f'{"포맷":<35} | {"평균 지연"} | {"FPS":<9} | P99')
print('-' * 70)

results = []
results.append(benchmark_torch(MODEL_PT, half=False))  # FP32
results.append(benchmark_torch(MODEL_PT, half=True))   # FP16

## 셀 4: ONNX 변환

### 왜 ONNX인가?
- PyTorch 그래프를 **프레임워크 독립 IR**로 직렬화
- onnxruntime이 그래프를 분석 → 사용 가능한 최적화 적용
- `opset=17`: ONNX 표준 연산자 집합 버전 (높을수록 최신 연산 지원)
- `dynamic=True`: 배치 크기/해상도가 실행 시 결정 (서빙에 유리)

In [ ]:
ONNX_PATH = Path('yolov8_optimized.onnx')

if not ONNX_PATH.exists():
    print('ONNX 변환 중...')
    model = YOLO(str(MODEL_PT))
    # Ultralytics 내장 export 활용
    exported = model.export(
        format='onnx',
        imgsz=640,
        opset=17,
        dynamic=False,   # 고정 크기 → TensorRT 최적화에 유리
        simplify=True,   # onnx-simplifier: 불필요한 노드 제거
        half=False,      # ONNX는 FP32로, TensorRT에서 FP16 처리
    )
    # 내보낸 파일을 현재 폴더로 복사
    import shutil
    src = Path(str(MODEL_PT).replace('.pt', '.onnx'))
    if src.exists():
        shutil.copy(src, ONNX_PATH)
    print(f'ONNX 저장 완료: {ONNX_PATH}')
else:
    print(f'기존 ONNX 사용: {ONNX_PATH}')

# 모델 크기 비교
pt_size   = MODEL_PT.stat().st_size / 1e6
onnx_size = ONNX_PATH.stat().st_size / 1e6 if ONNX_PATH.exists() else 0
print(f'\n모델 크기')
print(f'  PyTorch (.pt): {pt_size:.1f} MB')
print(f'  ONNX:          {onnx_size:.1f} MB')

## 셀 5: ONNX Runtime 벤치마크 (CPU / GPU)

In [ ]:
def benchmark_onnx(onnx_path, provider, n_warmup=WARMUP, n_repeat=REPEAT):
    """ONNX Runtime 벤치마크"""
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    try:
        sess = ort.InferenceSession(
            str(onnx_path),
            providers=[provider],
            sess_options=sess_opts,
        )
    except Exception as e:
        print(f'{provider} 사용 불가: {e}')
        return None

    input_name = sess.get_inputs()[0].name

    # 워밍업
    for _ in range(n_warmup):
        sess.run(None, {input_name: sample_np})

    # 측정
    latencies = []
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        sess.run(None, {input_name: sample_np})
        latencies.append((time.perf_counter() - t0) * 1000)

    lat_mean = np.mean(latencies)
    lat_p99  = np.percentile(latencies, 99)
    fps      = 1000 / lat_mean
    short    = provider.replace('ExecutionProvider', '')
    label    = f'ONNX Runtime ({short})'
    print(f'{label:<35} | {lat_mean:6.2f} ms | {fps:7.1f} FPS | P99: {lat_p99:.1f} ms')
    return {'label': label, 'latency_ms': lat_mean, 'fps': fps, 'p99_ms': lat_p99}

if ONNX_PATH.exists():
    r = benchmark_onnx(ONNX_PATH, 'CPUExecutionProvider')
    if r: results.append(r)

    if 'CUDAExecutionProvider' in ort.get_available_providers():
        r = benchmark_onnx(ONNX_PATH, 'CUDAExecutionProvider')
        if r: results.append(r)
    else:
        print('CUDAExecutionProvider 없음 → onnxruntime-gpu 설치 필요')
        print('  pip install onnxruntime-gpu')
else:
    print('ONNX 파일 없음 — 셀 4 먼저 실행')

## 셀 6: TensorRT 변환 및 벤치마크

### TensorRT가 빠른 이유
1. **레이어 퓨전**: Conv → BN → ReLU 3개 연산을 GPU 커널 1개로 합침
2. **정밀도 교정 (FP16)**: 텐서코어(RTX 4080의 경우 304개)를 풀로 활용
3. **커널 자동 선택**: GPU 아키텍처에 최적화된 CUDA 커널 자동 선택
4. **메모리 최적화**: 중간 텐서 메모리 재사용으로 대역폭 절감

In [ ]:
TRT_ENGINE_PATH = Path('yolov8_fp16.engine')

if TRT_AVAILABLE:
    if not TRT_ENGINE_PATH.exists():
        print('TensorRT FP16 엔진 빌드 중... (첫 빌드 2~5분 소요)')
        model = YOLO(str(MODEL_PT))
        model.export(format='engine', imgsz=640, half=True, device=0, workspace=4)
        src = Path(str(MODEL_PT).replace('.pt', '.engine'))
        if src.exists():
            import shutil; shutil.copy(src, TRT_ENGINE_PATH)
        print(f'엔진 저장: {TRT_ENGINE_PATH}')
    else:
        print(f'기존 엔진 사용: {TRT_ENGINE_PATH}')

    if TRT_ENGINE_PATH.exists():
        trt_model = YOLO(str(TRT_ENGINE_PATH))

        # TRT 엔진은 PyTorch 모듈이 아님 → predict()에 numpy 배열을 직접 전달
        # (이미지 파일 로드 오버헤드 제거, 전처리·추론·후처리만 측정)
        img_np = (sample_tensor.squeeze(0).permute(1,2,0).numpy() * 255).astype('uint8')

        # 워밍업
        for _ in range(WARMUP):
            trt_model.predict(img_np, verbose=False)
        torch.cuda.synchronize()

        # 측정
        latencies = []
        for _ in range(REPEAT):
            t0 = time.perf_counter()
            trt_model.predict(img_np, verbose=False)
            torch.cuda.synchronize()
            latencies.append((time.perf_counter() - t0) * 1000)

        lat_mean = np.mean(latencies)
        lat_p99  = np.percentile(latencies, 99)
        fps      = 1000 / lat_mean
        label    = 'TensorRT FP16 (GPU)'
        print(f'{label:<35} | {lat_mean:6.2f} ms | {fps:7.1f} FPS | P99: {lat_p99:.1f} ms')
        results.append({'label': label, 'latency_ms': lat_mean, 'fps': fps, 'p99_ms': lat_p99})

        # PyTorch도 동일하게 numpy → predict() 방식으로 재측정 (공정 비교용)
        pt_ref = YOLO(str(MODEL_PT))
        for _ in range(WARMUP):
            pt_ref.predict(img_np, verbose=False)
        torch.cuda.synchronize()

        lats_pt = []
        for _ in range(REPEAT):
            t0 = time.perf_counter()
            pt_ref.predict(img_np, verbose=False)
            torch.cuda.synchronize()
            lats_pt.append((time.perf_counter() - t0) * 1000)

        lat_pt = np.mean(lats_pt)
        print(f'\n[공정 비교 — numpy 입력, 동일 파이프라인]')
        print(f'  PyTorch FP32 predict(): {lat_pt:.2f} ms / {1000/lat_pt:.1f} FPS')
        print(f'  TensorRT FP16 predict(): {lat_mean:.2f} ms / {fps:.1f} FPS')
        print(f'  실질 속도 향상: {lat_pt/lat_mean:.2f}×')
else:
    print('TensorRT 미설치 — pip install tensorrt 후 재실행')

## 셀 7: 결과 시각화 및 분석

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if not results:
    print('측정 결과 없음 — 앞 셀 먼저 실행')
else:
    labels    = [r['label'] for r in results]
    fps_vals  = [r['fps'] for r in results]
    lat_vals  = [r['latency_ms'] for r in results]
    baseline  = fps_vals[0]  # PyTorch FP32 기준
    speedups  = [f / baseline for f in fps_vals]

    colors = []
    for r in results:
        if 'TensorRT' in r['label']:  colors.append('#E74C3C')
        elif 'FP16' in r['label']:    colors.append('#F39C12')
        elif 'CUDA' in r['label']:    colors.append('#2ECC71')
        else:                          colors.append('#3498DB')

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('YOLOv8 포맷별 추론 성능 비교', fontsize=15, fontweight='bold')

    # (1) FPS 비교
    bars = axes[0].barh(labels, fps_vals, color=colors, edgecolor='white', linewidth=0.5)
    axes[0].set_xlabel('FPS (높을수록 좋음)')
    axes[0].set_title('추론 속도 (FPS)')
    for bar, val in zip(bars, fps_vals):
        axes[0].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                     f'{val:.1f}', va='center', fontsize=10)
    axes[0].axvline(30, color='red', linestyle='--', alpha=0.5, label='실시간 기준 30 FPS')
    axes[0].legend(fontsize=9)

    # (2) 지연시간 비교
    bars2 = axes[1].barh(labels, lat_vals, color=colors, edgecolor='white', linewidth=0.5)
    axes[1].set_xlabel('지연시간 ms (낮을수록 좋음)')
    axes[1].set_title('추론 지연시간 (ms)')
    for bar, val in zip(bars2, lat_vals):
        axes[1].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                     f'{val:.1f}ms', va='center', fontsize=10)
    axes[1].axvline(33, color='red', linestyle='--', alpha=0.5, label='33ms = 30FPS 한계')
    axes[1].legend(fontsize=9)

    # (3) 속도 향상 배율
    bars3 = axes[2].barh(labels, speedups, color=colors, edgecolor='white', linewidth=0.5)
    axes[2].set_xlabel('속도 향상 배율 (PyTorch FP32 대비)')
    axes[2].set_title('상대적 속도 향상')
    for bar, val in zip(bars3, speedups):
        axes[2].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                     f'{val:.2f}×', va='center', fontsize=10)
    axes[2].axvline(1.0, color='gray', linestyle='--', alpha=0.5)

    legend_patches = [
        mpatches.Patch(color='#3498DB', label='PyTorch FP32 (기준)'),
        mpatches.Patch(color='#F39C12', label='PyTorch FP16'),
        mpatches.Patch(color='#2ECC71', label='ONNX GPU'),
        mpatches.Patch(color='#E74C3C', label='TensorRT FP16'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.05))
    plt.tight_layout()
    plt.savefig('benchmark_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('benchmark_result.png 저장 완료')

## 셀 8: 정확도 유지 확인 (mAP 비교)

In [ ]:
# 속도 향상이 정확도를 희생하지 않는지 확인
# 동일 이미지에 대해 PyTorch vs ONNX vs TRT 검출 결과 비교

import cv2
import matplotlib.pyplot as plt
import numpy as np

TEST_IMG = str(img_paths[5])

# PyTorch 결과
pt_model = YOLO(str(MODEL_PT))
pt_result = pt_model.predict(TEST_IMG, verbose=False)[0]

# ONNX 결과
onnx_results = None
if ONNX_PATH.exists():
    onnx_model = YOLO(str(ONNX_PATH))
    onnx_results = onnx_model.predict(TEST_IMG, verbose=False)[0]

# TRT 결과
trt_results = None
if TRT_AVAILABLE and TRT_ENGINE_PATH.exists():
    trt_model_eval = YOLO(str(TRT_ENGINE_PATH))
    trt_results = trt_model_eval.predict(TEST_IMG, verbose=False)[0]

# 시각화
n_cols = 1 + (onnx_results is not None) + (trt_results is not None)
fig, axes = plt.subplots(1, n_cols, figsize=(7 * n_cols, 7))
if n_cols == 1: axes = [axes]

def draw_boxes(result, title, ax):
    img = cv2.imread(str(TEST_IMG))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    boxes = result.boxes
    for box in boxes:
        x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
        conf = float(box.conf[0])
        cls  = int(box.cls[0])
        name = result.names[cls]
        cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(img, f'{name} {conf:.2f}', (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
    ax.imshow(img)
    ax.set_title(f'{title}\n검출 수: {len(boxes)}개')
    ax.axis('off')

draw_boxes(pt_result, 'PyTorch FP32', axes[0])
idx = 1
if onnx_results: draw_boxes(onnx_results, 'ONNX Runtime', axes[idx]); idx += 1
if trt_results:  draw_boxes(trt_results,  'TensorRT FP16', axes[idx])

plt.suptitle('포맷별 검출 결과 비교 (정확도 유지 확인)', fontsize=13)
plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('accuracy_comparison.png 저장 완료')

## 셀 9: 최종 요약 — 이력서용 수치

In [ ]:
print('=' * 65)
print('  YOLOv8 모델 최적화 — 최종 성능 요약')
print('=' * 65)

if results:
    baseline_fps = results[0]['fps']
    baseline_lat = results[0]['latency_ms']

    for r in results:
        speedup = r['fps'] / baseline_fps
        print(f"\n  [{r['label']}]")
        print(f"    FPS:      {r['fps']:.1f}")
        print(f"    지연시간:   {r['latency_ms']:.2f} ms")
        print(f"    P99 지연:  {r['p99_ms']:.1f} ms")
        print(f"    속도 향상:  {speedup:.2f}×")

    print()
    print('  [결론]')
    best = max(results, key=lambda x: x['fps'])
    best_speedup = best['fps'] / baseline_fps
    print(f"    최고 포맷:   {best['label']}")
    print(f"    최대 속도향상: {best_speedup:.1f}×")

    realtime_capable = [r for r in results if r['fps'] >= 30]
    print(f"    실시간(30FPS) 가능 포맷: {len(realtime_capable)}개")
    for r in realtime_capable:
        print(f"      - {r['label']}: {r['fps']:.1f} FPS")

print()
print('  [이력서 기재 예시]')
if len(results) >= 2:
    trt_r = next((r for r in results if 'TensorRT' in r['label']), results[-1])
    sp = trt_r['fps'] / baseline_fps
    print(f"    YOLOv8 모델을 TensorRT FP16으로 최적화하여")
    print(f"    추론 속도 {sp:.1f}× 향상 ({results[0]['fps']:.0f} → {trt_r['fps']:.0f} FPS)")
    print(f"    자율주행 실시간 처리(30FPS) 기준 달성")
print('=' * 65)